<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/evaluations/event_1/evento1_parteB_diagnostico.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/evaluations/event_1/evento1_parteB_diagnostico.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fevaluations%2Fevent_1%2Fevento1_parteB_diagnostico.ipynb)

# SI7011 · Evento evaluativo 1 · Parte B: diagnóstico de una receta
En parejas · entrega en una semana

Este notebook entrena un MLP sobre `digits` (imágenes de 8×8 de dígitos escritos a mano, 10 clases; viene con scikit-learn, sin descargas). Corre en CPU en segundos.

La receta tiene **exactamente dos fallas**: una de **código** (en las funciones de las secciones 3 y 4) y una de **configuración** (en `RECETA`). Tu tarea no es solo encontrarlas: es **demostrarlas con evidencia**.

**Para cada falla** completa la bitácora de la sección 7:

síntoma → medición → hipótesis → **predicción escrita antes de correr** → experimento que cambia **una sola cosa** (registrado en MLflow) → resultado → conclusión

Después corrige las dos, entrena la receta final (en validación: accuracy ≥ 95 % y pérdida ≤ 0.25), evalúa en prueba **una sola vez** y completa el despliegue mínimo.

**Entrega:** un `.zip` con este notebook ejecutado y `mlflow.db` (con `mlruns/`). Encontrar la falla leyendo el código no basta: la nota depende de la evidencia.

In [ ]:
%pip install -q mlflow

In [ ]:
import copy, json, math, os, time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import mlflow
os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
logging.getLogger('mlflow').setLevel(logging.WARNING)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· MLflow', mlflow.__version__)

## 1. Datos (listo)
Contrato: 64 columnas `pixel_0 … pixel_63`, enteros entre 0 y 16. Partición estratificada 60/20/20. El escalado es fijo (÷ 16), así que no se ajusta con ningún subconjunto.

In [ ]:
digits = load_digits(as_frame=True)
df = digits.frame
FEATURES = list(digits.feature_names)
CONTRACT = {'features': FEATURES, 'min': 0, 'max': 16, 'classes': list(range(10))}

def validate_rows(frame, contract=CONTRACT):
    missing = [c for c in contract['features'] if c not in frame.columns]
    if missing:
        raise ValueError(f'Faltan columnas: {missing[:3]}')
    X = frame[contract['features']].to_numpy(np.float32)
    if np.isnan(X).any() or X.min() < contract['min'] or X.max() > contract['max']:
        raise ValueError(f"Valores fuera de [{contract['min']}, {contract['max']}]")
    return X

X_all = validate_rows(df) / 16.0
y_all = df['target'].to_numpy(np.int64)
X_rest, X_te, y_rest, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=0)
X_train, X_val, X_test = map(torch.as_tensor, (X_tr, X_va, X_te))
y_train, y_val, y_test = map(torch.as_tensor, (y_tr, y_va, y_te))
json.dump({'fuente': 'sklearn.datasets.load_digits', 'filas': len(df), 'clases': np.bincount(y_all).tolist()},
          open('data_card.json', 'w'))
print('entrenamiento', tuple(X_train.shape), '· validación', tuple(X_val.shape), '· prueba', tuple(X_test.shape))

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for k, ax in enumerate(axes):
    ax.imshow(X_tr[k].reshape(8, 8), cmap='gray_r'); ax.set_title(int(y_tr[k]), fontsize=8); ax.axis('off')
plt.show()

## 2. La receta

In [ ]:
RECETA = dict(
    depth=3,
    width=128,
    act='relu',
    init='normal',
    dropout=0.2,
    optimizer='adamw',
    lr=0.001,
    momentum=0.0,
    weight_decay=0.0001,
    batch_size=64,
    epochs=40,
)

## 3. El modelo

In [ ]:
ACTS = {'relu': nn.ReLU, 'tanh': nn.Tanh, 'sigmoid': nn.Sigmoid}

def make_model(r):
    layers, n_in = [], 64
    for _ in range(r['depth']):
        layers += [nn.Linear(n_in, r['width']), ACTS[r['act']](), nn.Dropout(r['dropout'])]
        n_in = r['width']
    model = nn.Sequential(*layers, nn.Linear(n_in, 10))
    for m in model:
        if isinstance(m, nn.Linear):
            if r['init'] == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu'); nn.init.zeros_(m.bias)
            elif r['init'] == 'xavier':
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
            elif r['init'] == 'normal':
                nn.init.normal_(m.weight, 0, 1); nn.init.zeros_(m.bias)
            # 'pytorch': se deja la inicialización por defecto
    return model

def make_optimizer(model, r):
    if r['optimizer'] == 'sgd':
        return torch.optim.SGD(model.parameters(), lr=r['lr'], momentum=r['momentum'], weight_decay=r['weight_decay'])
    return torch.optim.AdamW(model.parameters(), lr=r['lr'], weight_decay=r['weight_decay'])

## 4. Entrenar y evaluar

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_epoch(model, opt, batch_size):
    model.train()
    perm = torch.randperm(len(X_train))
    total, norms = 0.0, []
    for i in range(0, len(X_train), batch_size):
        b = perm[i:i + batch_size]
        xb, yb = X_train[b], y_train[b]
        loss = loss_fn(model(xb), yb)
        loss.backward()
        norms.append(nn.utils.clip_grad_norm_(model.parameters(), float('inf')).item())
        opt.step()
        total += loss.item() * len(b)
    return total / len(X_train), float(np.mean(norms))

@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    logits = model(X)
    return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

## 5. Registro en MLflow (listo)
`run` entrena una receta y registra parámetros, pérdida inicial, curvas por época y norma del gradiente, y guarda `best.pt` (pesos + receta + contrato) con early stopping sobre `val_loss`. **Toda corrida que uses como evidencia debe pasar por `run`.**

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('evento1-parteB')

def run(name, r, notes=''):
    torch.manual_seed(0)
    model = make_model(r); opt = make_optimizer(model, r)
    hist = {'train_loss': [], 'grad_norm': [], 'val_loss': [], 'val_acc': []}
    with mlflow.start_run(run_name=name):
        mlflow.log_params(r); mlflow.set_tag('notas', notes)
        init_loss = evaluate(model, X_val, y_val)[0]
        mlflow.log_metric('init_loss', init_loss)
        best = (float('inf'), None, -1)
        for epoch in range(r['epochs']):
            tl, gn = train_epoch(model, opt, r['batch_size'])
            vl, va = evaluate(model, X_val, y_val)
            for k, v in zip(hist, (tl, gn, vl, va)):
                hist[k].append(v)
            if not math.isfinite(tl):
                mlflow.set_tag('estado', 'diverge'); break
            mlflow.log_metrics({'train_loss': tl, 'grad_norm': gn, 'val_loss': vl, 'val_acc': va}, step=epoch)
            if vl < best[0]:
                best = (vl, copy.deepcopy(model.state_dict()), epoch)
        if best[1] is not None:
            model.load_state_dict(best[1])
            vl, va = evaluate(model, X_val, y_val)
            mlflow.log_metrics({'best_val_loss': vl, 'best_val_acc': va, 'best_epoch': best[2]})
            torch.save({'receta': r, 'state_dict': best[1], 'contract': CONTRACT}, 'best.pt'); mlflow.log_artifact('best.pt')
    print(f"{name}: pérdida inicial {init_loss:.3f} · "
          f"train_loss final {hist['train_loss'][-1]:.3f} · mejor val_acc {max(hist['val_acc'], default=0):.3f}")
    return model, hist

def plot_runs(hists, keys=('train_loss', 'val_loss', 'val_acc', 'grad_norm')):
    fig, axes = plt.subplots(1, len(keys), figsize=(3.4 * len(keys), 2.8))
    for ax, k in zip(axes, keys):
        for name, h in hists.items():
            ax.plot(range(1, len(h[k]) + 1), h[k], label=name)
        ax.set_title(k); ax.set_xlabel('época')
        if k == 'grad_norm': ax.set_yscale('log')
    axes[0].legend(fontsize=7); fig.tight_layout(); plt.show()

## 6. Instrumentos de medición (listos)
Son los de los notebooks 1–4 de la sesión 2. Úsalos como evidencia.

| Función | Qué mide | Notebook |
|---|---|---|
| `activation_stds(model, x)` | std$(a_l)$ por capa | 1 |
| `grad_norms(model, x, y)` | $\lVert\nabla_{W_l}J\rVert$ por capa, tras un `backward` | 1 |
| `overfit_one_batch(model, opt)` | pérdida tras 300 pasos sobre los mismos 64 ejemplos | 4 |
| `weight_norm(model)` | $\lVert W\rVert$ | 3 |
| `evaluate(model, X, y)` dos veces seguidas | si la evaluación es determinista | 3 |

In [ ]:
def activation_stds(model, x):
    stds, hooks = [], [m.register_forward_hook(lambda mod, i, o: stds.append(o.std().item()))
                       for m in model if isinstance(m, (nn.ReLU, nn.Tanh, nn.Sigmoid))]
    with torch.no_grad():
        model.eval(); model(x)
    for h in hooks: h.remove()
    return stds

def grad_norms(model, x, y):
    model.train(); model.zero_grad(); loss_fn(model(x), y).backward()
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]

def overfit_one_batch(model, opt, n=64, steps=300):
    xb, yb = X_train[:n], y_train[:n]
    model.train()
    for _ in range(steps):
        loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

weight_norm = lambda model: sum(p.pow(2).sum() for n, p in model.named_parameters() if 'weight' in n).sqrt().item()

## 7. Corrida 0: la receta tal como está

In [ ]:
model0, h0 = run('0-receta-entregada', RECETA)
plot_runs({'receta entregada': h0})

## 8. Bitácora · falla 1

| Paso | Tu registro |
|---|---|
| **Síntoma** (qué viste, con números) | |
| **Medición** (qué instrumento usaste y qué dio) | |
| **Hipótesis** | |
| **Predicción** (escrita **antes** de correr el experimento) | |
| **Experimento** (qué **una** cosa cambiaste; nombre de la corrida en MLflow) | |
| **Resultado** | |
| **Conclusión** (¿es de código o de configuración? ¿qué línea o valor?) | |

In [ ]:
# Mediciones y experimentos de la falla 1. Nombra las corridas como '1-...'.


## 9. Bitácora · falla 2

| Paso | Tu registro |
|---|---|
| **Síntoma** (qué viste, con números) | |
| **Medición** (qué instrumento usaste y qué dio) | |
| **Hipótesis** | |
| **Predicción** (escrita **antes** de correr el experimento) | |
| **Experimento** (qué **una** cosa cambiaste; nombre de la corrida en MLflow) | |
| **Resultado** | |
| **Conclusión** (¿es de código o de configuración? ¿qué línea o valor?) | |

In [ ]:
# Mediciones y experimentos de la falla 2. Nombra las corridas como '2-...'.


## 10. Receta final
Corrige **las dos** fallas: el código en las secciones 3–4 y la configuración en una copia de `RECETA`. Luego entrena con `run('final', ...)`.

In [ ]:
# RECETA_FINAL = {**RECETA, ...}
# model_final, h_final = run('final', RECETA_FINAL)


In [ ]:
runs = mlflow.search_runs(experiment_names=['evento1-parteB'])
runs = runs.drop_duplicates('tags.mlflow.runName', keep='first').set_index('tags.mlflow.runName')
assert 'final' in runs.index, "Primero entrena la receta final con run('final', ...)."
final = runs.loc['final']
assert final['metrics.best_val_acc'] >= 0.95, 'La receta final debe llegar a 95 % de accuracy en validación.'
assert final['metrics.best_val_loss'] <= 0.25, 'La receta final debe tener pérdida de validación ≤ 0.25.'
runs[['metrics.init_loss', 'metrics.best_val_loss', 'metrics.best_val_acc']].round(3)

## 11. Prueba, una sola vez (listo)

In [ ]:
ckpt = torch.load(mlflow.artifacts.download_artifacts(run_id=final.run_id, artifact_path='best.pt'), weights_only=False)
model = make_model(ckpt['receta']); model.load_state_dict(ckpt['state_dict'])
test_loss, test_acc = evaluate(model, X_test, y_test)
with mlflow.start_run(run_id=final.run_id):
    mlflow.log_metrics({'test_loss': test_loss, 'test_acc': test_acc})
print(f'prueba: loss {test_loss:.3f} · acc {test_acc:.3f}')

## 12. Despliegue mínimo
Escribe `predict_raw(ckpt, frame)`: recibe un `DataFrame` con las 64 columnas **crudas** (0–16) y devuelve la clase predicha por fila. Usa solo lo que trae `ckpt`.

In [ ]:
@torch.no_grad()
def predict_raw(ckpt, frame):
    # contrato -> ÷ 16 -> modelo de ckpt['receta'] con sus pesos -> clase
    raise NotImplementedError

In [ ]:
raw = pd.DataFrame((X_te * 16).round().astype(int), columns=FEATURES)
pred = np.asarray(predict_raw(ckpt, raw))
acc_raw = (pred == y_te).mean()
print(f'1. Accuracy desde filas crudas: {acc_raw:.3f} (prueba: {test_acc:.3f})')
assert abs(acc_raw - test_acc) < 1e-6, 'El despliegue no reproduce la evaluación de prueba.'
bad = raw.head(2).copy(); bad.iloc[0, 0] = 99
try:
    predict_raw(ckpt, bad); raise AssertionError('Una fila inválida debió ser rechazada.')
except ValueError as e:
    print('2. Contrato: fila rechazada →', e)

## 13. Dos preguntas finales
Responde cada una citando **corridas de MLflow** por su nombre.

1. De tus dos fallas, ¿cuál habrías detectado **antes de entrenar**, solo con la pérdida inicial o con un instrumento de la sección 6? ¿Cuál solo se ve entrenando? Muestra el número que lo prueba.
2. Si hubieras corregido solo la falla de configuración, ¿qué habrías visto? Responde con una corrida, no con una suposición.

_Respuestas:_